# Preliminary Cross-Subject Evaluation

This notebook compares handcrafted EEG features and CSP + LDA using an initial subject-level split.

Subjects S016–S020 are held out from training in this experiment. However, they are later included in the development cohort used for further model selection.

The final independent evaluation is presented in Notebook 11.

## 1. Dataset Verification

We verify the availability of 60 motor imagery recordings from S001–S020.

In [6]:
from pathlib import Path

data_dir = Path("../data/raw")

subjects = [f"S{i:03d}" for i in range(1, 21)]
runs = ["04", "08", "12"]

expected_files = [data_dir / f"{subject}R{run}.edf" 
                  for subject in subjects for run in runs]

missing_files = [path.name for path in expected_files
                 if not path.is_file() or path.stat().st_size == 0]

print("Expected recordings:", len(expected_files))
print("Available recordings:", len(expected_files) - len(missing_files))
print("Missing recordings:", len(missing_files))

assert not missing_files, f"Missing EDF files: {missing_files}"

print("Dataset verification passed.")

Expected recordings: 60
Available recordings: 60
Missing recordings: 0
Dataset verification passed.


## 2. EEG Preprocessing and Epoch Extraction

EEG recordings from 20 subjects are preprocessed using the same parameters as in the previous experiments.

Signals are average-referenced, bandpass-filtered between 8–30 Hz, and segmented into 4-second motor imagery epochs (T1 and T2).

Both the full 64-channel representation and the C3, Cz, C4 subset are retained for subsequent model evaluation.

In [7]:
import mne
import numpy as np

motor_channels = ["C3..", "Cz..", "C4.."]

X_all_list = []
X_motor_list = []
y_list = []
groups_list = []

for subject in subjects:
    for run in runs:
        recording_id = f"{subject}R{run}"
        raw_path = data_dir / f"{recording_id}.edf"

        raw = mne.io.read_raw_edf(raw_path, preload=True, verbose=False)
        raw.set_eeg_reference(ref_channels="average", projection=False, verbose=False)
        raw.filter(l_freq=8.0, h_freq=30.0, picks="eeg", verbose=False)

        events, event_id = mne.events_from_annotations(raw, verbose=False)

        epochs = mne.Epochs(raw, events,  event_id={"T1": event_id["T1"], "T2": event_id["T2"]}, tmin=0.0, tmax=4.0, 
                            picks="eeg", baseline=None, preload=True, reject_by_annotation=True, verbose=False)

        X_all_list.append(epochs.get_data())
        X_motor_list.append(epochs.copy().pick(motor_channels).get_data())

        y_list.append(np.where(epochs.events[:, -1] == event_id["T1"], 0, 1))
        groups_list.extend([subject] * len(epochs))

        print(f"{recording_id}: {len(epochs)} epochs")

X_epochs_all = np.concatenate(X_all_list, axis=0)
X_epochs = np.concatenate(X_motor_list, axis=0)
y = np.concatenate(y_list)
groups = np.array(groups_list)

print("\nAll-channel shape:", X_epochs_all.shape)
print("Motor-channel shape:", X_epochs.shape)
print("Class distribution:", np.bincount(y))
print("Subjects:", len(np.unique(groups)))

S001R04: 15 epochs
S001R08: 15 epochs
S001R12: 15 epochs
S002R04: 15 epochs
S002R08: 15 epochs
S002R12: 15 epochs
S003R04: 15 epochs
S003R08: 15 epochs
S003R12: 15 epochs
S004R04: 15 epochs
S004R08: 15 epochs
S004R12: 15 epochs
S005R04: 15 epochs
S005R08: 15 epochs
S005R12: 15 epochs
S006R04: 15 epochs
S006R08: 15 epochs
S006R12: 15 epochs
S007R04: 15 epochs
S007R08: 15 epochs
S007R12: 15 epochs
S008R04: 15 epochs
S008R08: 15 epochs
S008R12: 15 epochs
S009R04: 15 epochs
S009R08: 15 epochs
S009R12: 15 epochs
S010R04: 15 epochs
S010R08: 15 epochs
S010R12: 15 epochs
S011R04: 15 epochs
S011R08: 15 epochs
S011R12: 15 epochs
S012R04: 15 epochs
S012R08: 15 epochs
S012R12: 15 epochs
S013R04: 15 epochs
S013R08: 15 epochs
S013R12: 15 epochs
S014R04: 15 epochs
S014R08: 15 epochs
S014R12: 15 epochs
S015R04: 15 epochs
S015R08: 15 epochs
S015R12: 15 epochs
S016R04: 15 epochs
S016R08: 15 epochs
S016R12: 15 epochs
S017R04: 15 epochs
S017R08: 15 epochs
S017R12: 15 epochs
S018R04: 15 epochs
S018R08: 15 

### Preprocessing Results

All 60 EEG recordings were successfully processed, producing 900 motor imagery epochs from 20 subjects.

The dataset contains 455 T1 (left-hand) and 445 T2 (right-hand) epochs, with a nearly balanced class distribution.

Both 64-channel and three-channel EEG representations are available for model evaluation.

## 3. Handcrafted Feature Extraction

Time-domain and frequency-domain features are extracted from the preprocessed C3, Cz, and C4 epochs using the same methodology as in the initial experiments.

The original 21 features are calculated, followed by the previously defined reduction to 15 features.

This ensures consistency with the handcrafted-feature LDA baseline.т

In [8]:
import pandas as pd
from scipy.signal import welch

sfreq = 160
feature_rows = []

for epoch_data in X_epochs:
    row = {}

    for channel_idx, channel_name in enumerate(motor_channels):
        signal = epoch_data[channel_idx]

        # Time-domain features
        row[f"{channel_name}_mean"] = np.mean(signal)
        row[f"{channel_name}_std"] = np.std(signal)
        row[f"{channel_name}_rms"] = np.sqrt(np.mean(signal ** 2))

        # Frequency-domain features
        freqs, psd = welch(signal, fs=sfreq, nperseg=256)

        alpha_mask = (freqs >= 8) & (freqs < 13)
        beta_mask = (freqs >= 13) & (freqs <= 30)
        df = freqs[1] - freqs[0]

        alpha_power = np.sum(psd[alpha_mask]) * df
        beta_power = np.sum(psd[beta_mask]) * df
        total_power = alpha_power + beta_power

        row[f"{channel_name}_alpha_power"] = alpha_power
        row[f"{channel_name}_beta_power"] = beta_power
        row[f"{channel_name}_relative_alpha_power"] = (alpha_power / total_power if total_power > 0 else np.nan)
        row[f"{channel_name}_relative_beta_power"] = (beta_power / total_power if total_power > 0 else np.nan)

    feature_rows.append(row)

features_df = pd.DataFrame(feature_rows)

features_to_drop = ["C3.._rms", "Cz.._rms", "C4.._rms", "C3.._relative_beta_power", "Cz.._relative_beta_power", "C4.._relative_beta_power"]

X_handcrafted = features_df.drop(columns=features_to_drop)

print("Original feature shape:", features_df.shape)
print("Reduced feature shape:", X_handcrafted.shape)
print("Missing values:", X_handcrafted.isna().sum().sum())
print("All features finite:", np.isfinite(X_handcrafted.to_numpy()).all())

assert X_handcrafted.shape == (900, 15)
assert np.isfinite(X_handcrafted.to_numpy()).all()

Original feature shape: (900, 21)
Reduced feature shape: (900, 15)
Missing values: 0
All features finite: True


## 4. Feature Consistency Check

We compare the newly extracted features for S001–S005 with the original processed dataset.

This verifies that feature extraction remains consistent with the previous experiments before final model evaluation.

In [10]:
original_df = pd.read_csv("../data/processed/eeg_features_all_subjects.csv")
original_binary = original_df[original_df["event"].isin(["T1", "T2"])].reset_index(drop=True)

feature_columns = features_df.columns.tolist()

X_original = original_binary[feature_columns].to_numpy()
X_new = features_df.iloc[:225][feature_columns].to_numpy()

y_original = original_binary["event"].map({"T1": 0, "T2": 1}).to_numpy()
groups_original = original_binary["subject_id"].to_numpy()

print("Original features:", X_original.shape)
print("New features:", X_new.shape)
print("Labels match:", np.array_equal(y_original, y[:225]))
print("Subjects match:", np.array_equal(groups_original, groups[:225]))

max_difference = np.max(np.abs(X_original - X_new))
print("Maximum feature difference:", f"{max_difference:.3e}")

assert np.array_equal(y_original, y[:225])
assert np.array_equal(groups_original, groups[:225])
np.testing.assert_allclose(X_original, X_new, rtol=1e-6, atol=1e-18)

print("Feature consistency check passed.")

Original features: (225, 21)
New features: (225, 21)
Labels match: True
Subjects match: True
Maximum feature difference: 8.327e-17
Feature consistency check passed.


### Feature Consistency Results

Feature extraction was consistent with the original dataset for S001–S005.

Event labels, subject identifiers, and feature dimensions matched exactly. The maximum numerical difference was negligible (8.327 × 10⁻¹⁷).

This confirms that the expanded dataset uses the same feature extraction methodology as the previous experiments.

## 5. Subject-Independent Evaluation Setup

The dataset is divided at the subject level into 15 training subjects (S001–S015) and 5 independent test subjects (S016–S020).

Leave-One-Subject-Out cross-validation is performed within the training set for model assessment and selection.

The final models are trained on all 15 training subjects and evaluated on S016–S020, which remain excluded from model selection.

Balanced accuracy is the primary metric, with Macro-F1 as a secondary metric.

In [12]:
train_subjects = [f"S{i:03d}" for i in range(1, 16)]
test_subjects = [f"S{i:03d}" for i in range(16, 21)]

train_mask = np.isin(groups, train_subjects)
test_mask = np.isin(groups, test_subjects)

assert not np.any(train_mask & test_mask)
assert np.all(train_mask | test_mask)
assert set(groups[train_mask]).isdisjoint(set(groups[test_mask]))

print("Training subjects:", len(np.unique(groups[train_mask])))
print("Test subjects:", len(np.unique(groups[test_mask])))

print("\nTraining epochs:", train_mask.sum())
print("Test epochs:", test_mask.sum())

print("\nTraining class distribution:", np.bincount(y[train_mask]))
print("Test class distribution:", np.bincount(y[test_mask]))

Training subjects: 15
Test subjects: 5

Training epochs: 675
Test epochs: 225

Training class distribution: [342 333]
Test class distribution: [113 112]


## 6. Training Set Cross-Validation

Handcrafted-feature LDA and CSP + LDA are evaluated using Leave-One-Subject-Out cross-validation on the 15 training subjects.

Both models use configurations established during the previous exploratory experiments.

Balanced accuracy and Macro-F1 are used to assess cross-subject classification performance before independent testing on S016–S020.

In [13]:
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import LeaveOneGroupOut, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_train_handcrafted = X_handcrafted.loc[train_mask]
X_train_csp = X_epochs[train_mask]

y_train = y[train_mask]
groups_train = groups[train_mask]

models = {"Handcrafted LDA": make_pipeline(StandardScaler(),
                                           LinearDiscriminantAnalysis()),
          "CSP + LDA": make_pipeline(CSP(n_components=3, reg=None, log=True, norm_trace=False, rank=None),
                                     LinearDiscriminantAnalysis())}

datasets = {"Handcrafted LDA": X_train_handcrafted,
            "CSP + LDA": X_train_csp}

scoring = {"accuracy": "accuracy",
           "balanced_accuracy": "balanced_accuracy",
           "f1_macro": "f1_macro"}

logo = LeaveOneGroupOut()
cv_results_all = {}
comparison_results = []

for model_name, model in models.items():
    cv_results = cross_validate(model, datasets[model_name], y_train, groups=groups_train, cv=logo, scoring=scoring)

    cv_results_all[model_name] = cv_results

    comparison_results.append({"Model": model_name,
                               "Accuracy": cv_results["test_accuracy"].mean(),
                               "Balanced Accuracy": cv_results["test_balanced_accuracy"].mean(),
                               "Macro-F1": cv_results["test_f1_macro"].mean(),
                               "BA Std": cv_results["test_balanced_accuracy"].std()})

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df.round(3))

Computing rank from data with rank=None
    Using tolerance 4.9e-06 (2.2e-16 eps * 3 dim * 7.3e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 4.9e-06 (2.2e-16 eps * 3 dim * 7.4e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 4.9e-06 (2.2e-16 eps * 3 dim * 7.3e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estim

,Model,Accuracy,Balanced Accuracy,Macro-F1,BA Std
0,Handcrafted LDA,0.542,0.541,0.510,0.080
1,CSP + LDA,0.596,0.594,0.539,0.138


### Training Cross-Validation Results

CSP + LDA achieved higher balanced accuracy (**0.594**) and Macro-F1 (**0.539**) than handcrafted-feature LDA (0.541 and 0.510, respectively).

However, CSP showed greater variability across subjects (balanced accuracy std: 0.138 vs. 0.080).

These results suggest a potential advantage of spatial filtering, but cross-subject performance remains inconsistent.

## 7. Subject-Level Model Comparison

We compare balanced accuracy and Macro-F1 for each held-out training subject.

This analysis helps determine whether CSP consistently outperforms handcrafted-feature LDA or whether its average advantage is driven by a small number of subjects.

The independent test subjects (S016–S020) remain excluded.

In [14]:
subject_order = [groups_train[test_idx][0] for _, test_idx in logo.split(X_train_csp, y_train, groups_train)]

subject_comparison = pd.DataFrame({"Subject": subject_order,
                                   "Handcrafted BA": cv_results_all["Handcrafted LDA"]["test_balanced_accuracy"],
                                   "CSP BA": cv_results_all["CSP + LDA"]["test_balanced_accuracy"],
                                   "Handcrafted Macro-F1": cv_results_all["Handcrafted LDA"]["test_f1_macro"],
                                   "CSP Macro-F1": cv_results_all["CSP + LDA"]["test_f1_macro"]})

subject_comparison["BA Difference"] = (subject_comparison["CSP BA"] - subject_comparison["Handcrafted BA"])

display(subject_comparison.round(3))

print("Subjects where CSP has higher BA:",
      (subject_comparison["BA Difference"] > 0).sum(), "of", len(subject_comparison))

,Subject,Handcrafted BA,CSP BA,Handcrafted Macro-F1,CSP Macro-F1,BA Difference
0,S001,0.641,0.638,0.636,0.612,-0.003
1,S002,0.545,0.500,0.432,0.338,-0.045
2,S003,0.537,0.554,0.525,0.554,0.018
3,S004,0.469,0.630,0.460,0.568,0.161
4,S005,0.485,0.506,0.405,0.418,0.021
5,S006,0.494,0.479,0.467,0.338,-0.015
6,S007,0.750,0.935,0.737,0.933,0.185
7,S008,0.490,0.607,0.488,0.564,0.117
8,S009,0.464,0.452,0.402,0.440,-0.012
9,S010,0.637,0.664,0.610,0.664,0.027


Subjects where CSP has higher BA: 9 of 15


### Subject-Level Comparison Results

CSP outperformed handcrafted-feature LDA in 9 of 15 training subjects.

The largest improvements were observed for S015, S007, S013, and S004, while performance decreased for several other participants.

Although CSP achieved higher mean balanced accuracy, substantial subject-level variability remains. Independent evaluation is required to assess whether this advantage generalizes to new subjects.

## 8. Preliminary Held-Out Evaluation

Both models are trained on S001–S015 and evaluated on S016–S020.

This experiment provides an initial assessment of cross-subject generalization. It is not the final independent test of the selected model.

In [16]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

test_results = []
test_predictions = {}

for model_name, model in models.items():
    X_data = X_handcrafted if model_name == "Handcrafted LDA" else X_epochs

    X_train = X_data[train_mask]
    X_test = X_data[test_mask]
    y_test = y[test_mask]
    groups_test = groups[test_mask]

    final_model = clone(model)
    final_model.fit(X_train, y_train)

    y_pred = final_model.predict(X_test)
    test_predictions[model_name] = y_pred

    for subject in np.unique(groups_test):
        mask = groups_test == subject

        test_results.append({"Model": model_name,
                             "Subject": subject,
                             "Accuracy": accuracy_score(y_test[mask], y_pred[mask]),
                             "Balanced Accuracy": balanced_accuracy_score(y_test[mask], y_pred[mask]),
                             "Macro-F1": f1_score(y_test[mask], y_pred[mask], average="macro", zero_division=0)})

test_results_df = pd.DataFrame(test_results)

display(test_results_df.round(3))

Computing rank from data with rank=None
    Using tolerance 5e-06 (2.2e-16 eps * 3 dim * 7.6e+09  max singular value)
    Estimated rank (data): 3
    data: rank 3 computed from 3 data channels with 0 projectors
Reducing data rank from 3 -> 3
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.


,Model,Subject,Accuracy,Balanced Accuracy,Macro-F1
0,Handcrafted LDA,S016,0.556,0.548,0.500
1,Handcrafted LDA,S017,0.489,0.478,0.328
2,Handcrafted LDA,S018,0.511,0.500,0.338
3,Handcrafted LDA,S019,0.578,0.571,0.533
4,Handcrafted LDA,S020,0.467,0.473,0.418
5,CSP + LDA,S016,0.600,0.594,0.564
6,CSP + LDA,S017,0.511,0.500,0.338
7,CSP + LDA,S018,0.422,0.425,0.415
8,CSP + LDA,S019,0.600,0.598,0.595
9,CSP + LDA,S020,0.578,0.576,0.574


### Preliminary Held-Out Results

CSP + LDA achieved higher balanced accuracy than handcrafted-feature LDA for four of five independent test subjects.

However, CSP performance varied across subjects, ranging from 0.425 (S018) to 0.598 (S019).

Although CSP showed a modest advantage, both approaches demonstrated limited cross-subject generalization.

## 9. Preliminary Performance Summary

We summarize model performance across the five independent test subjects and compare it with training-set cross-validation results.

Mean balanced accuracy, Macro-F1, and between-subject variability are reported to assess generalization to unseen participants.

In [18]:
final_summary = test_results_df.groupby("Model", sort=False).agg(
    Accuracy=("Accuracy", "mean"),
    Balanced_Accuracy=("Balanced Accuracy", "mean"),
    Macro_F1=("Macro-F1", "mean"),
    BA_Std=("Balanced Accuracy", "std")).reset_index()

final_summary = final_summary.rename(columns={
    "Balanced_Accuracy": "Balanced Accuracy",
    "Macro_F1": "Macro-F1",
    "BA_Std": "BA Std"})

display(final_summary.round(3))

,Model,Accuracy,Balanced Accuracy,Macro-F1,BA Std
0,Handcrafted LDA,0.520,0.514,0.423,0.044
1,CSP + LDA,0.542,0.539,0.497,0.075


### Preliminary Performance Results

CSP + LDA achieved higher independent-test balanced accuracy (**0.539**) and Macro-F1 (**0.497**) than handcrafted-feature LDA (0.514 and 0.423, respectively).

Both models performed worse than during training-set cross-validation, indicating limited generalization to unseen subjects.

Although CSP demonstrated a modest advantage, its performance varied more across test subjects. The small test population limits the reliability of this comparison.

## 10. Confusion Matrix and Error Analysis

Confusion matrices are used to examine classification errors on the independent test set.

We compare predictions for left-hand (T1) and right-hand (T2) motor imagery to identify possible class-specific biases in both models.

In [19]:
from sklearn.metrics import confusion_matrix, classification_report

y_test = y[test_mask]

for model_name, y_pred in test_predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    print(f"\n{model_name} — Confusion Matrix")

    cm_df = pd.DataFrame(cm, index=["Actual T1", "Actual T2"], columns=["Predicted T1", "Predicted T2"])

    display(cm_df)

    print("Classification Report:")
    print(classification_report(y_test, y_pred, labels=[0, 1], target_names=["T1 (Left)", "T2 (Right)"], zero_division=0))


Handcrafted LDA — Confusion Matrix


,Predicted T1,Predicted T2
Actual T1,51,62
Actual T2,46,66


Classification Report:
              precision    recall  f1-score   support

   T1 (Left)       0.53      0.45      0.49       113
  T2 (Right)       0.52      0.59      0.55       112

    accuracy                           0.52       225
   macro avg       0.52      0.52      0.52       225
weighted avg       0.52      0.52      0.52       225


CSP + LDA — Confusion Matrix


,Predicted T1,Predicted T2
Actual T1,73,40
Actual T2,63,49


Classification Report:
              precision    recall  f1-score   support

   T1 (Left)       0.54      0.65      0.59       113
  T2 (Right)       0.55      0.44      0.49       112

    accuracy                           0.54       225
   macro avg       0.54      0.54      0.54       225
weighted avg       0.54      0.54      0.54       225



### Confusion Matrix Results

The independent test evaluation revealed different class-specific error patterns.

Handcrafted-feature LDA achieved higher recall for T2 (0.59) than T1 (0.45), while CSP + LDA showed the opposite pattern, with higher recall for T1 (0.65) than T2 (0.44).

CSP correctly classified 122 of 225 epochs, compared with 117 for handcrafted LDA.

Although CSP performed slightly better overall, neither model demonstrated reliable discrimination between left- and right-hand motor imagery.

Pooled classification metrics differ from subject-averaged metrics because they are calculated across all test epochs rather than averaged across individual participants.

## Conclusion

This experiment compared handcrafted EEG features and CSP + LDA using a preliminary subject-level split.

CSP + LDA achieved a mean balanced accuracy of 0.539, compared with 0.514 for handcrafted-feature LDA.

Although CSP performed better on average, performance varied considerably across subjects.

These results motivated further experiments with channel selection, temporal windows, and alternative model architectures.

The subjects used for this evaluation were later included in the development cohort. The final independent evaluation is reported in Notebook 11.